In [1]:
from sur_puch import *

model_path = "/default-vepfs/public/user/ga/Iron/models/Llama-3.1-8B"
tokenizer, model, device = load_model(model_path)


/root/miniconda3/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/root/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


🚨 `javis_all_layer_kvs` is part of LlamaModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/miniconda3/lib/python3.10/site-packages/transformers/models/llama/modeling_llama.py.
🚨 `javis_meta` is part of LlamaModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/miniconda3/lib/python3.10/site-packages/transformers/models/llama/modeling_llama.py.
🚨 `javis_all_layer_kvs` is part of LlamaForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/miniconda3/lib/python3.10/site-packages/transformers/models/llama/modeling_llama.py.
🚨 `javis_meta` is part of LlamaForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/miniconda3/lib/python3.10/site-packages/transformers/models/llama/modeling_llama.py.


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 129.36it/s]


In [ ]:

text = """The Ise-class battleships were a pair of dreadnought battleships built for the Imperial Japanese Navy during World War I. Originally intended to be repeats of the preceding Fusō class, they were redesigned before construction began."""
needle = "The secret passcode is tiger."
import json
with open("seg_text.jsonl", "r", encoding="utf-8") as f:
    # 每一行 load 一次，形成一个包含多个字典的列表
    data_list = [json.loads(line) for line in f if line.strip()]
text = data_list[0]['text'].replace(" @-@ ", "-").replace(" @,@ ", ",")


input_ids, ns, ne, tokens, meta = build_input_ids(
    tokenizer, text, needle=needle, insert_pos=30
)

surprisal = compute_surprisal(model, input_ids, device=device)

print_local_window(tokens, surprisal, ns, ne, left=8, right=8)

segments = segment_surprisal_boundary_aware(
    surprisal=surprisal,
    tokens=tokens,
    min_len=8,
    max_len=24,
    acc_threshold=20.0,
    roll_window=3,
    event_threshold=6.0,
    event_min_span=7,
    search_window=6,
    punct_bonus=2.0,
    local_min_bonus=1.0,
)

print_segments(tokens, segments, ns, ne)
print(analyze_needle_coverage(segments, ns, ne))


Local window: [23, 46)

 idx           token  surprisal     mark
  23              se      0.021         
  24              -g      0.076         
  25             ata      0.000         
  26            Ġsen      0.582         
  27             kan      0.002         
  28              Ġ)      0.883         
  29           Ġwere      0.200         
  30              Ġa      0.777         
  31             The     17.375   needle
  32         Ġsecret     11.188   needle
  33           Ġpass     11.562   needle
  34            code      4.875   needle
  35             Ġis      2.828   needle
  36          Ġtiger     10.688   needle
  37               .      1.930   needle
  38           Ġpair      4.906         
  39             Ġof      0.028         
  40          Ġdread      1.586         
  41               n      0.020         
  42           ought      0.027         
  43        Ġbattles      0.410         
  44            hips      0.051         
  45          Ġbuilt      0.201 

: 